# Qwen Image 2.1 — NORMAL (25 adım) — MANUEL (ComfyUI arayüzünde elle) — Colab

ComfyUI'yi tünelle açar, grafiği **sen elle sürersin**. Bu bir *ilk bakış*: Qwen Image 2.1 çalışıyor mu, nasıl görünüyor — Queen Editor'e yeni bir çizim tarzı olarak girip girmeyeceği burada görülecek (v10 yol haritası, madde 482). Uygulamanın koduna dokunmaz.

**Turbo (8 adım) ayrı defterde: `turbo.ipynb`.** İkisi ayrı runtime'da koşar, her biri yalnız kendi modelini indirir.

**Input:** prompt (ComfyUI UI'da) · **Output:** resim

**Gerekenler:** **A100** runtime. Colab **Secrets**'ta `HF_TOKEN` varsa kullanılır; yoksa da iner (dosyalar açık).

Sıra:
1. **CONFIG** — GPU
2. **Ortak yardımcılar** — log + fail-loud run + model doğrulama
3. **ComfyUI** — `master`'dan; custom node yok, Qwen Image 2.1 ComfyUI'nin kendinde
4. **Modeller** — ~26,8 GB; önce boş disk
5. **Resmî şablon** — `image_qwen_image_2_1_t2i`, ComfyUI'nin Workflows paneline
6. **Başlat + cloudflared tünel** → UI linki

> Drive kullanılmaz; modeller her oturumda kaynaktan iner (Colab geçici diski). Üretilen resimler de orada durur: beğendiğini UI'dan indir, runtime kapanınca gider.
>
> **Runtime → Change runtime type → A100.**
>
> **Run all** → en alttaki linke gir → sol kenarda **Workflows** → `image_qwen_image_2_1_t2i` → prompt'u yaz → **Run** (Ctrl+Enter).
>
> Hangi dosya neden iniyor: **`indirilecekler.md`** · nasıl açılır: **`instructions.md`**

## 1) CONFIG

Doldurulacak bir şey yok → **Run all**.

`HF_TOKEN` Colab **Secrets**'tan okunur, değeri hiçbir yere basılmaz. Yoksa indirme token'sız gider: dosyalar açık, ama HF token'sız isteklere 429 dönebiliyor.

GPU burada basılır; GPU yoksa hücre durur.

**Drive mount yok.** Bu deneme Drive kullanmıyor; standardın "önce Drive mount" kuralı Drive kullanan defterler için.

In [ ]:
# === CONFIG ===
COMFY_PORT = 8188
COMFY_ROOT = "/content/ComfyUI"
COMFY_LOG  = "/content/comfyui.log"

import subprocess
from google.colab import userdata

# Optional: every file is public, but HF can answer 429 to requests without a token. Read once here
# and handed to every download, so huggingface_hub never goes to the vault itself mid-download (in
# queen-editor that late ask timed out and H3 came down unauthenticated into a 429, madde 437).
try:
    HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
except Exception as e:
    HF_TOKEN = ""
    print(f"⚠️  HF_TOKEN okunamadı — {type(e).__name__}: {e}\n"
          "    Token'sız inecek: dosyalar açık, ama HF 429 dönebilir.")
if HF_TOKEN:
    print("✓ HF_TOKEN: Secrets'tan okundu")

# === GPU ===
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)
except FileNotFoundError:
    smi = None
if smi is None or smi.returncode != 0 or not smi.stdout.strip():
    said = (smi.stderr or smi.stdout).strip() if smi else "nvidia-smi bulunamadı"
    raise RuntimeError(f"❌ GPU yok — Runtime → Change runtime type → A100\n{said}")
print(f"\n=== GPU ===\n{smi.stdout.strip()}")

## 2) Ortak Yardımcılar

Tek yerde tanımlanır, sonraki hücreler bunları kullanır:

- `log(msg, level)` — konsola durum satırı
- `run(cmd, label)` — **tek komut-hatası kapısı**: exit ≠ 0 veya timeout → `RuntimeError` + komutun **gerçek stderr** kuyruğu
- `check_safetensors(path)` — **ok** / **partial** / **invalid**. Beklenen boyutu **dosyanın kendi header'ından** hesaplar (tensör `data_offsets`); sunucuya sormaz
- `head_text(path)` — dosyanın ilk byte'ları, **ham**: bozuk bir indirmede orada ne olduğunu gösterir

Kanonik defterden *(`loop_maker/comfy_ui.ipynb`, H3 denemesi üzerinden)* birebir geliyor. H3'ün `strip_unreferenced_tail`'i burada yok: o damga H3'ün nicelemelerini üreten araçtan geliyordu, bu dosyalar Comfy-Org'un kendi deposundan.

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
import os, json, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌", "SKIP": "⏭️ "}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr."""
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (NOTEBOOK-STANDARD, section 3).
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors)")

## 3) ComfyUI — `master`'dan

Qwen Image 2.1'in node'ları (`TextEncodeQwenImage21`, `QwenImage21Cache`) **ComfyUI'nin kendinde**: [PR #16400](https://github.com/Comfy-Org/ComfyUI/pull/16400), 19 Eylül 2026. Custom node da Manager da gerekmiyor; şablonun bütün node'ları çekirdekten.

ComfyUI her koşuda `master`'dan çekilir, sonra klonda `TextEncodeQwenImage21` aranır: yoksa hücre durur — eski bir ComfyUI, grafiği açınca "missing node" olarak değil burada görünür.

`hf_xet` de burada kurulur: HF dosyaları Xet deposunda, ve `hf_hub_download` onunla parçaları doğrudan depodan, paralel çeker. Adresle (`resolve/…`) inen dosya HF'in köprüsünden geçiyor, o da sunucularının çoğunda ~8,7 MB/s'e düşüyor *(queen-editor, madde 438; xet-core #821)*.

Bir komut başarısız olursa hücre `RuntimeError` ile durur ve komutun kendi çıktısını basar.

In [ ]:
# === ComfyUI from master ===
import os

if not os.path.isdir(COMFY_ROOT):
    log("ComfyUI klonlanıyor")
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_ROOT],
        "clone ComfyUI", timeout=600)
run(["git", "pull", "-q"], "git pull ComfyUI", cwd=COMFY_ROOT)
log("ComfyUI gereksinimleri kuruluyor (birkaç dakika)")
run(["pip", "install", "-q", "-r", "requirements.txt"], "pip install ComfyUI", cwd=COMFY_ROOT)
log("hf_xet kuruluyor")
run(["pip", "install", "-q", "-U", "hf_xet"], "pip install hf_xet")

# Qwen Image 2.1 support is in ComfyUI's own comfy_extras/nodes_qwen.py (PR 16400). Looked for here,
# so a clone without it stops the run now rather than as a missing node in the UI.
commit = run(["git", "log", "-1", "--format=%h %cd"], "git log", cwd=COMFY_ROOT).strip()
QWEN_NODES = f"{COMFY_ROOT}/comfy_extras/nodes_qwen.py"
if not os.path.exists(QWEN_NODES) or "TextEncodeQwenImage21" not in open(QWEN_NODES).read():
    raise RuntimeError(f"❌ Bu ComfyUI'de TextEncodeQwenImage21 yok ({QWEN_NODES}, commit {commit})")
log(f"ComfyUI hazır — {commit}, Qwen Image 2.1 node'ları var", "OK")

## 4) Modeller — ~26,8 GB

Hepsi HF'te açık, [Comfy-Org/Qwen-Image-2.1](https://huggingface.co/Comfy-Org/Qwen-Image-2.1) deposundan: şablonun kendi `models` listesinin istediği dosyalar ve adresler bunlar. Boyutlar HF'in dosya listesinden, bayt bayt.

| Klasör | Dosya | Boyut | |
|---|---|---|---|
| `diffusion_models/` | `qwen_image_2.1_int8_convrot.safetensors` | 7,26 GB | normal model (DiT) |
| `text_encoders/` | `qwen3vl_8b_int8_convrot.safetensors` | 9,35 GB | metin kodlayıcı, Turbo'yla ortak |
| `vae/` | `qwen_image_2.1_vae_bf16.safetensors` | 0,68 GB | VAE, Turbo'yla ortak |
| `text_encoders/` | `qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors` | 9,47 GB | prompt geliştirici — aşağıda neden |

**Prompt geliştirici neden iniyor.** Şablon onu kapalı açıyor (`refine_prompt: false`), ve kapalıyken hiç yüklenmiyor: arada tembel bir anahtar var. Ama ComfyUI bir işi başlatmadan önce çıktıya bağlı **her** yükleyicinin dosyasını listede arıyor, tembel dal da dahil. Dosya yoksa **Run** `Value not in list` ile duruyor. Şablon dokunulmadan çalışsın diye iniyor.

Önce **boş disk**: inecek dosyaların toplamı + 5 GiB pay yoksa hiçbir şey inmeden durur.

Her dosya `hf_hub_download` ile `/content/hf_stage` altına iner, **doğrulanır** (header'daki boyut + HF listesindeki bayt sayısı), sonra gerçek yerine taşınır: ComfyUI yarım bir dosyayı gerçek adıyla görmez. Bozuk dosya **silinmez**, yerinde kalır ve hücre başını basıp durur. **Yeniden denemeden önce o dosyayı elle sil** (hata yolunu basar, çoğu zaman `/content/hf_stage` altında): silinmezse `hf_hub_download` aynı dosyayı yeniden verir ve hücre aynı yerde durur. Yarıda kalan bir koşuda yerine oturmuş dosyalar atlanır.

In [ ]:
import os, glob, shutil, time
from huggingface_hub import hf_hub_download

# === What this notebook downloads ===
# The files and folders are the ones the official template's own "models" list names (section 5
# checks that against the template it fetches). Sizes are the bytes HF's file list gives: the disk
# check adds them up, and each download is held to its own.
QWEN = "Comfy-Org/Qwen-Image-2.1"
MODELS = [
    # (repo, path in the repo, ComfyUI models folder, bytes, label)
    (QWEN, "diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
     "diffusion_models", 7_256_783_064, "Qwen Image 2.1 DiT int8"),
    (QWEN, "text_encoders/qwen3vl_8b_int8_convrot.safetensors",
     "text_encoders", 9_350_798_360, "Qwen3-VL 8B metin kodlayıcı int8"),
    (QWEN, "vae/qwen_image_2.1_vae_bf16.safetensors",
     "vae", 675_509_688, "VAE bf16"),
    # Off in the template and never loaded while refine_prompt is off -- the switch in front of it is
    # lazy. But ComfyUI validates every loader upstream of an output before it runs anything, lazy
    # branch or not (execution.py, validate_inputs), so without the file Run stops at
    # "Value not in list". Downloaded so the official template runs as it is.
    (QWEN, "text_encoders/qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors",
     "text_encoders", 9_471_072_252, "Prompt geliştirici Qwen3.5 9B int8"),
]

# hf_hub_download writes the repo's folders and its own .cache under local_dir; neither belongs among
# ComfyUI's models, so a file lands here and is moved into place -- a rename on one disk, not a copy.
STAGE = "/content/hf_stage"
# GiB the disk check keeps free past the models themselves.
HEADROOM_GIB = 5

def target_of(path, folder):
    return f"{COMFY_ROOT}/models/{folder}/{os.path.basename(path)}"

def settled(path, size, label):
    """The verdict's message, or a RuntimeError showing how the file starts. A bad file is never
    passed on and never deleted: it stays where it is, for inspection -- and since hf_hub_download
    hands a file already in STAGE straight back, a re-run needs it deleted by hand first."""
    state, msg = check_safetensors(path)
    if state == "ok" and os.path.getsize(path) != size:
        state, msg = "invalid", f"{os.path.getsize(path):,} bayt, HF listesi {size:,} diyor"
    if state != "ok":
        raise RuntimeError(f"❌ {label}: {state} — {msg}\n{path}\n--- file head ---\n{head_text(path)}\n"
                           f"Yeniden denemeden önce bu dosyayı elle sil: {path}")
    return msg

def hf_fetch(repo, path, folder, size, label):
    target = target_of(path, folder)
    if os.path.exists(target):
        log(f"{label}: zaten var ({settled(target, size, label)})")
        return
    log(f"{label}: iniyor ({human(size)})")
    start = time.perf_counter()
    try:
        # token=False, not None: None sends huggingface_hub to the vault on its own.
        got = hf_hub_download(repo, path, local_dir=STAGE, token=HF_TOKEN or False)
    except Exception as e:
        said = f"{type(e).__name__}: {e}"
        response = getattr(e, "response", None)
        if response is not None:
            try:
                body = response.text
            except Exception as read_error:   # a streamed body nobody read can raise when asked for
                body = f"(gövde okunamadı — {type(read_error).__name__}: {read_error})"
            said += f"\n--- response: HTTP {response.status_code} ---\n{body}"
        raise RuntimeError(f"❌ {label}: HF {repo}/{path}\n{said}") from None
    msg = settled(got, size, label)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    os.replace(got, target)
    took = time.perf_counter() - start
    log(f"{label}: indirildi ve doğrulandı — {took:.0f} sn, {size / took / 2**20:.1f} MB/s ({msg})", "OK")

# === Free disk, before anything downloads ===
need = sum(size for _, path, folder, size, _ in MODELS if not os.path.exists(target_of(path, folder)))
free = shutil.disk_usage("/content").free
log(f"İnecek: {need / 1024**3:.1f} GiB | Diskte boş: {free / 1024**3:.1f} GiB")
if free < need + HEADROOM_GIB * 1024**3:
    raise RuntimeError(f"❌ Disk yetmiyor: {need / 1024**3:.1f} GiB model + {HEADROOM_GIB} GiB pay "
                       f"gerekiyor, {free / 1024**3:.1f} GiB boş.")

# === Downloads ===
for repo, path, folder, size, label in MODELS:
    hf_fetch(repo, path, folder, size, label)

# === Summary (reaching here means every model downloaded + validated) ===
print("")
for folder in sorted({folder for _, _, folder, _, _ in MODELS}):
    print(f"📂 {folder}/")
    for f in sorted(glob.glob(f"{COMFY_ROOT}/models/{folder}/*.safetensors")):
        print(f"   {human(os.path.getsize(f)):>8s}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## 5) Resmî şablon — Workflows paneline

Grafik elle yazılmadı ve üretilmedi: Comfy-Org'un resmî şablonu, kaynağından **olduğu gibi** iner ve ComfyUI'nin kullanıcı workflow klasörüne konur. UI'da sol kenardaki **Workflows** panelinde tek tıkla açılır.

- Kaynak: https://github.com/Comfy-Org/workflow_templates/blob/main/templates/image_qwen_image_2_1_t2i.json
- Ham dosya: https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/image_qwen_image_2_1_t2i.json
- Hedef: `/content/ComfyUI/user/default/workflows/image_qwen_image_2_1_t2i.json`

Bayt bayt aynı yazılır; okunur ama değiştirilmez. Konmadan önce iki kontrol: dosya JSON mu, ve şablonun `models` listesinde adı geçen **her** dosya 4. bölümde inmiş mi. Şablon bir gün başka bir dosya isterse burada durur ve hangisi olduğunu söyler — UI'da "missing model" olarak değil.

Hedefte dosya zaten varsa hiçbir şey indirilmez ve dosyaya dokunulmaz: UI'da aynı adla kaydettiğin bir grafik, hücre tekrar çalışınca ezilmez.

In [ ]:
# === Official template -> ComfyUI's user workflows ===
import os, json

TEMPLATE = "image_qwen_image_2_1_t2i.json"
TEMPLATE_URL = f"https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/{TEMPLATE}"
# Where ComfyUI keeps the user's saved workflows; the UI's Workflows panel lists this folder.
WORKFLOWS = f"{COMFY_ROOT}/user/default/workflows"

def named_models(node):
    """Every {"name", "directory"} entry in the template -- the loaders' own "models" lists."""
    if isinstance(node, dict):
        if isinstance(node.get("name"), str) and isinstance(node.get("directory"), str):
            yield node["directory"], node["name"]
        for value in node.values():
            yield from named_models(value)
    elif isinstance(node, list):
        for value in node:
            yield from named_models(value)

saved = f"{WORKFLOWS}/{TEMPLATE}"
# A file already there may be the user's own save under the same name: a re-run leaves it alone.
if os.path.exists(saved):
    log(f"Şablon zaten var, olduğu gibi bırakıldı: Workflows → {TEMPLATE[:-5]} ({saved})")
else:
    os.makedirs(WORKFLOWS, exist_ok=True)
    part = saved + ".part"
    try:
        run(["curl", "-sS", "-L", "--fail-with-body", "--max-time", "60", "-o", part, TEMPLATE_URL],
            f"şablon {TEMPLATE}", timeout=120)
        with open(part, "rb") as f:
            graph = json.loads(f.read())
    except (RuntimeError, ValueError) as e:
        raise RuntimeError(f"❌ {e}\n{TEMPLATE_URL}\n--- dosyanın başı ---\n{head_text(part)}") from None

    wanted = sorted(set(named_models(graph)))
    missing = [f"models/{folder}/{name}" for folder, name in wanted
               if not os.path.exists(f"{COMFY_ROOT}/models/{folder}/{name}")]
    if not wanted or missing:
        raise RuntimeError(f"❌ Şablonun istediği dosyalar inenlerle tutmuyor ({TEMPLATE_URL})\n"
                           f"Şablonun istediği: {wanted or '(model listesi bulunamadı)'}\n"
                           f"Diskte olmayan: {missing}")
    # The bytes as they came, renamed into place only after both checks.
    os.replace(part, saved)
    log(f"Şablon hazır: Workflows → {TEMPLATE[:-5]} ({len(wanted)} model dosyası, hepsi diskte)", "OK")

## 6) Başlat + cloudflared Tünel

ComfyUI arka planda başlar (90 sn içinde `/system_stats` cevap vermezse log'un son 30 satırını basıp durur), tünel linki basılır, sonra hücre **bilerek açık kalır** — biterse Colab runtime'ı boşta sayıp tüneli öldürür. ComfyUI'nin canlı log'u bu hücrede akar.

In [ ]:
import subprocess, time, urllib.request, re, os

if not os.path.isfile("/content/cloudflared"):
    run(["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], "cloudflared")
    run(["chmod", "+x", "/content/cloudflared"], "chmod cloudflared")

# Re-run safety: kill the previous ComfyUI + tunnel before starting new ones
subprocess.run(["pkill", "-f", "main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(COMFY_LOG, "w")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
                 cwd=COMFY_ROOT, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(COMFY_LOG).readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")
log(f"ComfyUI ayakta ({(i + 1) * 2}s)", "OK")

# cloudflared output goes to a file, not a pipe: an unread pipe fills up and blocks the process.
tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read()) if os.path.exists(tunlog) else None
    if m:
        link = m.group(0)
        break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 ComfyUI linki: {link}\n")
print("⬆️  Linke gir → sol kenarda Workflows → image_qwen_image_2_1_t2i")
print("    positive_prompt'a yaz → Run (Ctrl+Enter)")
print("\n⚠️  cfg 1'de negatif prompt hiçbir şey yapmaz — şablonun kendi notu\n")

# === Keep the cell OPEN (critical) ===
# ComfyUI + the tunnel run in the background. If this cell ENDS, Colab can call the runtime idle
# and disconnect -> ComfyUI + link die. Streaming the log keeps the cell in the foreground and
# shows generation progress when Run is pressed in the UI.
# To stop: interrupt this cell (■) or Runtime -> Disconnect.
print("📡 ComfyUI çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", COMFY_LOG])
except KeyboardInterrupt:
    log("Hücre durduruldu — ComfyUI hâlâ arka planda çalışıyor (yeni link için bu hücreyi tekrar çalıştır).", "WARN")